In [18]:
from dotenv import load_dotenv
import os
from openai import OpenAI

load_dotenv(override=True)

client = OpenAI(
    api_key=os.getenv("MOONSHOT_API_KEY"),
    base_url=os.getenv("MOONSHOT_BASE_URL"),
)

In [5]:
message_test = """
-第一轮
    ("user", "你好，今天北京天气怎么样？"),
    ("assistant", "北京今天晴天，25°C，适合出行。需要我推荐景点吗？"),
-第二轮
    ("user", "推荐几个编程学习资源"),
    ("assistant", "推荐：1. GitHub开源项目 2. LeetCode刷题 3. MDN文档"),
-第三轮
    ("user", "能帮我写一段Python代码吗？实现快速排序"),
    ("assistant", "```python\ndef quicksort(arr):\n    if len(arr) <= 1: return arr\n    pivot = arr[len(arr)//2]\n    left = [x for x in arr if x < pivot]\n    middle = [x for x in arr if x == pivot]\n    right = [x for x in arr if x > pivot]\n    return quicksort(left) + middle + quicksort(right)\n```"),
-第四轮
    ("user", "这段代码的时间复杂度是多少？"),
    ("assistant", "快速排序平均时间复杂度O(n v1.0-log n)，最坏情况O(n²)"),
-第五轮
    ("user", "如何优化最坏情况？"),
    ("assistant", "可以使用随机化pivot选择，或改用三路快排处理重复元素"),
"""

case_prompt = """
case 1
用户没有提供参数 默认N为100轮
case 2
用户提交非法参数就是非数字类型参数 ，给一个提示 但是也保留100轮
case 3
用户提交<0的参数 给一个提示 但是也保留100轮
case 4
用户提交 >0 但是 <3 的参数 ,给一个提示，保留3轮对话
case 5
用户提交>3的参数 ，根据用户提交的参数保留对应的对话
"""
prompt = f"""
请按照我的需求帮我实现 openai编程多轮对话的管理上下文的代码
这里是需求
-这是一个基于滑动窗口的方式只保存用户最近N轮对话的上下文
-每轮对话需要包含 用户的content 和 模型返回的 content
-根据用户提交的参数（N）来保留最近的N轮

只需要给我最后的代码是需要使用openai的api方式可以直接运行的 并且需要包含 对应的运行方式 其他内容不允许输出，你需要验证通过之后再给我代码部分
验证逻辑如下
这里是模拟的用户和 大模型的N轮对话
{message_test}

以下是case
{case_prompt}
"""
messages = [{"role": "user", "content": prompt}]

response = client.chat.completions.create(
    model="kimi-k2.5",
    messages=messages
)
print(response.choices[0].message.content)

```python
import openai
from typing import List, Dict, Union, Optional

class ConversationManager:
    def __init__(self, api_key: str, max_rounds: Optional[Union[int, str]] = None):
        self.client = openai.OpenAI(api_key=api_key)
        self.messages: List[Dict[str, str]] = []
        self.max_rounds = self._validate_max_rounds(max_rounds)
    
    def _validate_max_rounds(self, max_rounds) -> int:
        """验证并处理max_rounds参数"""
        # Case 1: 无参数，默认100轮
        if max_rounds is None:
            return 100
        
        # Case 2: 非法参数（非数字类型）
        try:
            n = int(max_rounds)
        except (ValueError, TypeError):
            print("提示：参数必须是数字类型，已使用默认值100")
            return 100
        
        # Case 3: 参数<0
        if n < 0:
            print("提示：参数不能为负数，已使用默认值100")
            return 100
        
        # Case 4: 0 < 参数 < 3
        if 0 < n < 3:
            print("提示：参数小于3，已调整为保留3轮对话")
            return 3
        
        # Case 5: 参数>=3
        return 

In [17]:
import openai
from typing import List, Dict, Union, Optional


class ConversationManager:
    def __init__(self, client, max_rounds: Optional[Union[int, str]] = None):
        self.client = client
        self.messages: List[Dict[str, str]] = []
        self.max_rounds = self._validate_max_rounds(max_rounds)

    def _validate_max_rounds(self, max_rounds) -> int:
        """验证并处理max_rounds参数"""
        # Case 1: 无参数，默认100轮
        if max_rounds is None:
            return 100

        # Case 2: 非法参数（非数字类型）
        try:
            n = int(max_rounds)
        except (ValueError, TypeError):
            print("提示：参数必须是数字类型，已使用默认值100")
            return 100

        # Case 3: 参数<0
        if n < 0:
            print("提示：参数不能为负数，已使用默认值100")
            return 100

        # Case 4: 0 < 参数 < 3
        if 0 < n < 3:
            print("提示：参数小于3，已调整为保留3轮对话")
            return 3

        # Case 5: 参数>=3
        return n

    def add_message(self, role: str, content: str):
        """添加消息并应用滑动窗口"""
        self.messages.append({"role": role, "content": content})
        self._apply_sliding_window()

    def _apply_sliding_window(self):
        """保留最近N轮对话（每轮包含user和assistant两条消息）"""
        # 分离system消息（不参与滑动窗口计算）
        system_msgs = [m for m in self.messages if m["role"] == "system"]
        dialog_msgs = [m for m in self.messages if m["role"] != "system"]

        # 保留最近N轮对话（2*N条消息）
        keep_count = self.max_rounds * 2
        if len(dialog_msgs) > keep_count:
            dialog_msgs = dialog_msgs[-keep_count:]

        self.messages = system_msgs + dialog_msgs

    def chat(self, user_content: str, model: str = "kimi-k2.5") -> str:
        """执行对话并自动管理上下文"""
        # 添加用户消息
        self.add_message("user", user_content)

        # 调用OpenAI API
        response = self.client.chat.completions.create(
            model=model,
            messages=self.messages
        )

        # 获取助手回复并添加到历史
        assistant_content = response.choices[0].message.content
        self.add_message("assistant", assistant_content)

        return assistant_content

    def get_history(self) -> List[Dict[str, str]]:
        """获取当前对话历史"""
        return self.messages.copy()

    def set_system_prompt(self, content: str):
        """设置系统提示词（不受滑动窗口影响）"""
        # 移除旧的system消息
        self.messages = [m for m in self.messages if m["role"] != "system"]
        # 插入到开头
        self.messages.insert(0, {"role": "system", "content": content})

    def clear_history(self):
        """清空对话历史（保留system消息）"""
        self.messages = [m for m in self.messages if m["role"] == "system"]


# ==================== 运行方式 ====================
if __name__ == "__main__":

    conv = ConversationManager(client=client)
    response1 = conv.chat("你好，今天上海天气怎么样？")
    response2 = conv.chat("推荐几个编程学习资源 只需要告诉我名称就行")
    response3 = conv.chat("你好我的昵称是 404，每天都想睡觉，喜欢打羽毛球")
    response4 = conv.chat("你觉得今年小米公司发展的怎么样 给我几个关键字就行")
    history = conv.get_history()
    for i, msg in enumerate(conv.get_history(), 1):
        preview = msg['content'][:40] + "..." if len(msg['content']) > 40 else msg['content']
        print(f"{i}. {msg['role']}: {preview}")

    print('-'*30)

    conv2 = ConversationManager(client=client, max_rounds=20)
    for i, msg in enumerate(conv.get_history(), 1):
        conv2.add_message(msg['role'],msg['content'])
    q1 = conv2.chat("你记得我刚刚哪个城市的天气吗")
    print(q1)
    for i, msg in enumerate(conv2.get_history(), 1):
        preview = msg['content'][:40] + "..." if len(msg['content']) > 40 else msg['content']
        print(f"{i}. {msg['role']}: {preview}")

    print('-'*30)
    conv3 = ConversationManager(client=client, max_rounds=3)
    for i, msg in enumerate(conv2.get_history(), 1):
        conv3.add_message(msg['role'],msg['content'])
    q2 = conv3.chat("你还记得我我的昵称吗")
    print(q2)
    for i, msg in enumerate(conv3.get_history(), 1):
        preview = msg['content'][:40] + "..." if len(msg['content']) > 40 else msg['content']
        print(f"{i}. {msg['role']}: {preview}")

    print('-'*30)
    conv4 = ConversationManager(client=client, max_rounds=3)
    for i, msg in enumerate(conv3.get_history(), 1):
        conv4.add_message(msg['role'],msg['content'])
    q3 = conv4.chat("将我刚刚问到的公司发展情况的是什么？")
    print(q3)
    for i, msg in enumerate(conv4.get_history(), 1):
        preview = msg['content'][:40] + "..." if len(msg['content']) > 40 else msg['content']
        print(f"{i}. {msg['role']}: {preview}")


1. user: 你好，今天上海天气怎么样？
2. assistant: 我无法提供实时天气信息，因为我没有接入当前的气象数据系统。

建议您通过以下方式...
3. user: 推荐几个编程学习资源 只需要告诉我名称就行
4. assistant: freeCodeCamp  
Codecademy  
LeetCode  
G...
5. user: 你好我的昵称是 404，每天都想睡觉，喜欢打羽毛球
6. assistant: 你好 404！这个昵称挺有意思的（是来自那个 "Not Found" 的状态码吗...
7. user: 你觉得今年小米公司发展的怎么样 给我几个关键字就行
8. assistant: 小米SU7、人车家生态、高端化突破、业绩 resurgence、AI大模型、出海...
------------------------------
上海。
1. user: 你好，今天上海天气怎么样？
2. assistant: 我无法提供实时天气信息，因为我没有接入当前的气象数据系统。

建议您通过以下方式...
3. user: 推荐几个编程学习资源 只需要告诉我名称就行
4. assistant: freeCodeCamp  
Codecademy  
LeetCode  
G...
5. user: 你好我的昵称是 404，每天都想睡觉，喜欢打羽毛球
6. assistant: 你好 404！这个昵称挺有意思的（是来自那个 "Not Found" 的状态码吗...
7. user: 你觉得今年小米公司发展的怎么样 给我几个关键字就行
8. assistant: 小米SU7、人车家生态、高端化突破、业绩 resurgence、AI大模型、出海...
9. user: 你记得我刚刚哪个城市的天气吗
10. assistant: 上海。
------------------------------
404。
1. user: 你觉得今年小米公司发展的怎么样 给我几个关键字就行
2. assistant: 小米SU7、人车家生态、高端化突破、业绩 resurgence、AI大模型、出海...
3. user: 你记得我刚刚哪个城市的天气吗
4. assistant: 上海。
5. user: 你还记得我我的昵称吗
6. assistant: 404。
